# Graph Diameter & Isovists (GRID_SIZE=1.0)
Separate notebook for compute-intensive analyses at coarser grid resolution.

In [ ]:
import time
import numpy as np

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Grid import Grid
from matplotlib.path import Path as MplPath

renderer = "vscode"
print(Helper.Version())

In [ ]:
from pathlib import Path

HERE = Path.cwd()
ASSET_DIR  = HERE.parent / '02_graph_analysis' / 'assets'

OBJ_L1     = ASSET_DIR / 'TheNarkomfinHouse-01.obj'
OBJ_L2     = ASSET_DIR / 'TheNarkomfinHouse-02.obj'
OBJ_STAIRS = ASSET_DIR / 'TheNarkomfinHouse-Ktype-withStairs.obj'

GRID_SIZE     = 1.0
FLOOR_HEIGHTS = [0, 3]
FLOOR_NAMES   = ['L1 (Y=0)', 'L2 (Y=3)']
FLOOR_Z_VIS   = [0, 9]

In [ ]:
def points_inside_faces(face_list, test_pts):
    inside = np.zeros(len(test_pts), bool)
    for f in face_list:
        vs = Topology.Vertices(f)
        poly = np.array([(Vertex.X(v), Vertex.Y(v)) for v in vs])
        path = MplPath(poly)
        inside |= path.contains_points(test_pts)
    return inside

def rk(u, v):
    return (round(float(u), 3), round(float(v), 3))

def make_cell_face(cx, cy, cz, h):
    pts = [Vertex.ByCoordinates(cx - h, cy - h, cz),
           Vertex.ByCoordinates(cx + h, cy - h, cz),
           Vertex.ByCoordinates(cx + h, cy + h, cz),
           Vertex.ByCoordinates(cx - h, cy + h, cz)]
    return Face.ByWire(Wire.ByVertices(pts, close=True))

def show_ortho(fig):
    fig.update_layout(
        scene_camera=dict(
            eye=dict(x=1.6, y=-1.6, z=1.2),
            up=dict(x=0, y=0, z=1),
            projection=dict(type='orthographic')
        ),
        scene=dict(aspectmode='data'),
        autosize=True,
        margin=dict(l=10, r=10, t=10, b=10)
    )
    fig.show(renderer=renderer)

print('Utilities loaded.')

## Load floors, grid sample, build graph

In [ ]:
# Load floor faces
floor_faces = {}
for lv, obj_path, name in zip(FLOOR_HEIGHTS, [OBJ_L1, OBJ_L2], FLOOR_NAMES):
    result = Topology.ByOBJPath(str(obj_path))
    faces = []
    if isinstance(result, list):
        for item in result:
            if Topology.IsInstance(item, 'Cluster'):
                cf = Topology.Faces(item)
                if cf: faces.extend(cf)
            elif Topology.IsInstance(item, 'Face'):
                faces.append(item)
    else:
        faces = Topology.Faces(result)
    floor_faces[lv] = faces
    print(f'{name}: {len(faces)} faces')

# Stairs
result_stairs = Topology.ByOBJPath(str(OBJ_STAIRS))
all_stair_faces = []
if isinstance(result_stairs, list):
    for item in result_stairs:
        if Topology.IsInstance(item, 'Cluster'):
            cf = Topology.Faces(item)
            if cf: all_stair_faces.extend(cf)
        elif Topology.IsInstance(item, 'Face'):
            all_stair_faces.append(item)
else:
    all_stair_faces = Topology.Faces(result_stairs)

stair_surfaces = [f for f in all_stair_faces if len(set(round(Vertex.Z(v), 1) for v in Topology.Vertices(f))) > 1]
stair_locations = []
for f in stair_surfaces:
    verts = Topology.Vertices(f)
    xs = [Vertex.X(v) for v in verts]
    ys = [Vertex.Y(v) for v in verts]
    stair_locations.append(((min(xs)+max(xs))/2, (min(ys)+max(ys))/2))
print(f'Stairs: {len(stair_surfaces)}')

In [ ]:
# Grid sample
all_xs, all_ys = [], []
for lv in FLOOR_HEIGHTS:
    for f in floor_faces[lv]:
        for v in Topology.Vertices(f):
            all_xs.append(Vertex.X(v))
            all_ys.append(Vertex.Y(v))
UMIN, UMAX = min(all_xs), max(all_xs)
VMIN, VMAX = min(all_ys), max(all_ys)

us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])

floor_valid = {}
for lv, name in zip(FLOOR_HEIGHTS, FLOOR_NAMES):
    mask = points_inside_faces(floor_faces[lv], GRID_PTS)
    floor_valid[lv] = GRID_PTS[mask]
    print(f'  {name}: {len(floor_valid[lv])} nodes')

In [ ]:
# Build graph
all_v, all_e = [], []
floor_index_map = {}
cell_lookup = {}
H = GRID_SIZE / 2.0

def find_closest_node(node_xy, x, y):
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

for fi, lv in enumerate(FLOOR_HEIGHTS):
    valid = floor_valid[lv]
    z_vis = FLOOR_Z_VIS[fi]
    idx = {}
    for (u, v) in valid:
        key = rk(u, v)
        idx[key] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z_vis)))
        cell_lookup[(fi, key)] = make_cell_face(float(u), float(v), float(z_vis), H)
    floor_index_map[lv] = idx
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))

# Stair edges
l1_idx = floor_index_map[FLOOR_HEIGHTS[0]]
l2_idx = floor_index_map[FLOOR_HEIGHTS[1]]
for sx, sy in stair_locations:
    i1 = find_closest_node(floor_valid[FLOOR_HEIGHTS[0]], sx, sy)
    i2 = find_closest_node(floor_valid[FLOOR_HEIGHTS[1]], sx, sy)
    k1 = rk(floor_valid[FLOOR_HEIGHTS[0]][i1, 0], floor_valid[FLOOR_HEIGHTS[0]][i1, 1])
    k2 = rk(floor_valid[FLOOR_HEIGHTS[1]][i2, 0], floor_valid[FLOOR_HEIGHTS[1]][i2, 1])
    all_e.append(Edge.ByVertices([all_v[l1_idx[k1]], all_v[l2_idx[k2]]]))

t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f'Graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)')

## Graph Diameter

In [ ]:
t0 = time.time()
diameter = Graph.Diameter(building_graph)
print(f'Graph diameter: {diameter} steps  ({time.time()-t0:.1f}s)')
print(f'Graph density:  {Graph.Density(building_graph):.6f}')
print(f'Total vertices: {len(gverts)}')
print(f'Total edges:    {len(gedges)}')

## Visibility Graph Analysis / Isovists (L1 only)
Compute isovists from a coarse grid of viewpoints on L1.

In [ ]:
gallery = floor_faces[FLOOR_HEIGHTS[0]][0]

b_r = Wire.BoundingRectangle(gallery)
bd = Topology.Dictionary(b_r)
w = Dictionary.ValueAtKey(bd, 'width')
h = Dictionary.ValueAtKey(bd, 'length')
iso_grid = Grid.VerticesByDistances(gallery, clip=True,
                                     uRange=list(range(0, int(w)+10, 10)),
                                     vRange=list(range(0, int(h)+10, 10)))
iso_verts = Topology.Vertices(iso_grid)
print(f'Isovist viewpoints: {len(iso_verts)}')

l1_gverts = [v for v in gverts if round(Vertex.Z(v)) == FLOOR_Z_VIS[0]]

t0 = time.time()
iso_results = []
for i, v in enumerate(iso_verts):
    isovist = Face.Isovist(gallery, v)
    if isovist:
        b_list = Vertex.IsInternal2D(l1_gverts, isovist)
        n = sum(1 for b in b_list if b)
        d = Dictionary.ByKeyValue('visibility', n)
        v = Topology.SetDictionary(v, d)
        iso_results.append((v, n))
    if (i + 1) % 5 == 0:
        print(f'  {i+1}/{len(iso_verts)} done ({time.time()-t0:.0f}s)')
print(f'Isovists computed in {time.time()-t0:.1f}s')

In [ ]:
# Interpolate visibility to L1 graph vertices and render heatmap
iso_new_verts = [r[0] for r in iso_results]
vis_values = [r[1] for r in iso_results]
for v in l1_gverts:
    Vertex.InterpolateValue(v, vertices=iso_new_verts, n=2, key='visibility')

mn_v, mx_v = min(vis_values), max(vis_values)
l1_vis_cells = []
for v in l1_gverts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, 'visibility')
    if vb is None: vb = 0
    key = rk(Vertex.X(v), Vertex.Y(v))
    cell = cell_lookup.get((0, key))
    if cell is not None:
        col = Color.AnyToHex(Color.ByValueInRange(float(vb), minValue=mn_v, maxValue=mx_v, colorScale='thermal'))
        cd = Topology.Dictionary(cell)
        cd = Dictionary.SetValueAtKey(cd, 'vis_color', col)
        Topology.SetDictionary(cell, cd)
        l1_vis_cells.append(cell)

print(f'Visibility heatmap: {len(l1_vis_cells)} cells, range [{mn_v}, {mx_v}]')
fig = Topology.Show(l1_vis_cells,
              showFigure=False,
              faceColorKey='vis_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              backgroundColor='black',
              width=900, height=600,
              renderer=renderer)
show_ortho(fig)